<div style="display:flex; justify-content:center; align-items:center; gap:40px; margin-bottom:20px;">
  <img src="assets/logos/logo_next.png" alt="Logo Next Educacion" style="height:70px;" />
  <img src="assets/logos/logo_isabel_i.png" alt="Logo Universidad Isabel I" style="height:70px;" />
</div>

# Trabajo Practico - Asignatura 7
## Modelado Predictivo con Machine Learning

**Master en Big Data & Business Intelligence**  
**Curso:** 2025/2026  
**Grupo:** 1  
**Integrantes:**  
Jose Daniel Alfaro Alfaro  
Daniela Corona Miguel  
Feeby Mendez Villalobos  
Marco Ramirez Gomez  
**Fecha:** En Madrid, a 28 de marzo del 2026

## 1. Introduccion y contexto

Este notebook presenta una ejecucion reproducible del proyecto de la Asignatura 7, alineada con el enunciado y conectada con el enfoque del TFM del grupo. El trabajo integra informacion socioeconomica e institucional para analizar patrones estructurales de gobernanza.

Aunque el TFM se centra en arquitectura y modelado de datos relacionales, en este ejercicio se incorpora una capa analitica complementaria con Machine Learning para:

- construir un dataset maestro consistente,
- entrenar modelos supervisados para predecir `ge_2023` y `cc_2023`,
- y aplicar tecnicas no supervisadas (PCA + K-Means) para segmentar paises por perfiles estructurales.

El objetivo practico es combinar rigor tecnico, trazabilidad de ejecucion y una interpretacion sustantiva de resultados.

## 2. Objetivo del problema predictivo

**Objetivo general:** desarrollar un proyecto de Machine Learning aplicado al analisis de la gobernanza institucional a partir de variables estructurales socioeconomicas e institucionales.

**Objetivos especificos:**

- realizar EDA del dataset maestro 2023 para identificar distribuciones, correlaciones y posibles problemas de calidad de datos,
- preparar variables con imputacion de faltantes (mediana), estandarizacion y particion train/test,
- construir y evaluar modelos supervisados (Regresion Lineal y Random Forest) para `ge_2023` y `cc_2023`,
- interpretar resultados mediante metricas (MAE, RMSE, R2), validacion cruzada e importancia de variables,
- aplicar PCA + K-Means para detectar patrones latentes y agrupar paises por perfiles estructurales.

La conexion con el TFM se mantiene como extension analitica: este cuaderno no sustituye la arquitectura de datos del TFM, sino que aporta evidencia cuantitativa complementaria para interpretar gobernanza.

## 3. Configuracion del entorno

In [1]:
from pathlib import Path
import subprocess
import sys
import platform
import pandas as pd

BASE_DIR = Path.cwd()
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

print("Sistema operativo:", platform.platform())
print("Python:", sys.executable)
print("Base:", BASE_DIR)
print("Raw:", RAW_DIR)
print("Processed:", PROCESSED_DIR)

Sistema operativo: Linux-6.17.0-19-generic-x86_64-with-glibc2.42
Python: /home/jalfaro/Documents/Universidad/Asignatura 7. Modelado Predictivo con Machine Learning/Codigo/.venv/bin/python
Base: /home/jalfaro/Documents/Universidad/Asignatura 7. Modelado Predictivo con Machine Learning/Codigo
Raw: /home/jalfaro/Documents/Universidad/Asignatura 7. Modelado Predictivo con Machine Learning/Codigo/data/raw
Processed: /home/jalfaro/Documents/Universidad/Asignatura 7. Modelado Predictivo con Machine Learning/Codigo/data/processed


## 4. Datos y fuentes

Los archivos fuente deben estar en `data/raw`:

- `wgidataset-2025.xlsx`
- `CPI2023_Global_Results__Trends.xlsx`
- `worldbank_gdp.xlsx`
- `fsi.xlsx`
- `wdi.csv`

In [2]:
required_raw = [
    "wgidataset-2025.xlsx",
    "CPI2023_Global_Results__Trends.xlsx",
    "worldbank_gdp.xlsx",
    "fsi.xlsx",
    "wdi.csv",
]

missing = [f for f in required_raw if not (RAW_DIR / f).exists()]
if missing:
    raise FileNotFoundError(f"Faltan archivos en data/raw: {missing}")

print("OK: disponibilidad completa de datos fuente")

OK: disponibilidad completa de datos fuente


## 5. Metodologia de ejecucion

Se reutilizan los scripts existentes del proyecto (sin alterar su logica de negocio), ejecutandolos en secuencia desde este cuaderno.

In [3]:
def run_script(script_name: str):
    print(f"\n>>> Ejecutando: {script_name}")
    subprocess.run([sys.executable, script_name], check=True)

def run_many(scripts):
    for s in scripts:
        run_script(s)

### 5.1 Preprocesamiento

In [4]:
preprocess_scripts = [
    "WGI rev00.py",
    "WGI CC rev00.py",
    "WGI GC rev00.py",
    "CPI ver00.py",
    "PIB test ver00.py",
    "WDI rev00.py",
    "fsi ver00.py",
]

run_many(preprocess_scripts)


>>> Ejecutando: WGI rev00.py
Columnas detectadas:
Index(['ID variable (economy code/ gov. dimension/ year)', 'Economy (name)',
       'Economy (code)', 'Region', 'Income classification', 'Year',
       'Governance dimension', 'Number of sources',
       'Governance estimate (approx. -2.5 to +2.5)',
       'Standard error (estimate)',
       'Lower threshold (90% conf. int. estimate)',
       'Upper threshold (90% conf. int. estimate)', 'Governance score (0-100)',
       'Standard error (gov. score)', 'Lower threshold (90% conf. int. score)',
       'Upper threshold (90% conf. int. score)', 'ADB mean', 'AFR mean',
       'ARB mean', 'ASB mean', 'ASD mean', 'BPS mean', 'BTI mean', 'CCR mean',
       'EBR mean', 'EIU mean', 'EQI mean', 'EUB mean', 'FRH mean', 'GCB mean',
       'GCS mean', 'GII mean', 'GWP mean', 'HER mean', 'HRM mean', 'HUM mean',
       'IFD mean', 'IJT mean', 'IRP mean', 'LBO mean', 'MSI mean', 'OBI mean',
       'PIA mean', 'PRS mean', 'RSF mean', 'VAB mean', 'VDM me

### 5.2 Integracion de datos (dataset maestro)

In [5]:
run_script("Merged_all_DS_rev00.py")

master_path = PROCESSED_DIR / "dataset_maestro_2023.csv"
df_master = pd.read_csv(master_path)
print("Shape dataset maestro:", df_master.shape)
df_master.head()


>>> Ejecutando: Merged_all_DS_rev00.py
Tamaños iniciales:
WDI: (265, 9)
WGI_GE: (213, 2)
WGI_CC: (215, 2)
WGI_GC: (215, 8)
CPI: (180, 2)
GDP: (247, 2)
FSI: (168, 2)

Tamaño después de todos los merges: (158, 21)

Primeras filas del dataset maestro:
  iso3  electricity_access  internet_users  gdp_per_capita  education_secondary  education_tertiary  ...   rl_2023  cc_2023_composite  governance_composite  cpi_2023           gdp  fsi_2023
0  AFG                85.3         17.7089      413.757895                  NaN                 NaN  ... -1.896805          -1.198754             -1.841324        20  1.715223e+10     106.6
1  AGO                51.1         44.7581     2916.136633            51.483905           10.049358  ... -1.194443          -0.624252             -0.814307        33  1.071677e+11      86.9
2  ALB               100.0         83.1356     9730.869219            96.136877           64.729352  ... -0.153011          -0.464044             -0.045352        37  2.349124e+10 

,iso3,electricity_access,internet_users,gdp_per_capita,education_secondary,education_tertiary,health_expenditure,unemployment,life_expectancy,ge_2023,...,va_2023,pv_2023,ge_2023_composite,rq_2023,rl_2023,cc_2023_composite,governance_composite,cpi_2023,gdp,fsi_2023
0,AFG,85.3,17.7089,413.757895,NaN,NaN,14.985763,14.008,66.035,-2.092459,...,-2.037082,-2.392283,-2.092459,-1.430561,-1.896805,-1.198754,-1.841324,20,1.715223e+10,106.6
1,AGO,51.1,44.7581,2916.136633,51.483905,10.049358,2.546929,14.051,64.617,-0.828703,...,-1.020936,-0.313165,-0.828703,-0.904342,-1.194443,-0.624252,-0.814307,33,1.071677e+11,86.9
2,ALB,100.0,83.1356,9730.869219,96.136877,64.729352,7.052403,10.669,79.602,0.293429,...,-0.036394,0.024657,0.293429,0.063251,-0.153011,-0.464044,-0.045352,37,2.349124e+10,56.8
3,ARE,100.0,100.0000,49850.687220,101.944972,61.345611,4.970951,2.151,82.909,1.310084,...,-0.907704,0.782952,1.310084,1.105352,0.449289,0.991858,0.621972,68,5.226222e+11,37.0
4,ARG,100.0,89.2290,14261.846570,105.574584,107.822574,10.269518,6.139,77.395,0.061473,...,0.496160,-0.020767,0.061473,-0.506513,-0.237774,-0.354078,-0.093583,37,6.494617e+11,46.4


## 6. Analisis exploratorio

El EDA se realiza con el script `EDA rev00.py` y cubre cuatro bloques principales:

1. **Distribuciones univariantes:** deteccion de asimetrias, concentraciones y dispersion por variable.
2. **Boxplots por grupos tematicos:** comparacion por familias (socioeconomicas, WGI, riesgo/corrupcion) para evitar distorsiones de escala.
3. **Matriz de correlacion:** identificacion de relaciones lineales relevantes entre variables explicativas y targets.
4. **Scatterplots con regresion:** contraste visual de relaciones bivariantes clave para `ge_2023` y `cc_2023`.

Hallazgos generales consistentes con la literatura y con el documento de resultados:

- correlacion positiva entre GE/CC y variables de desarrollo (PIB per capita, conectividad, esperanza de vida),
- correlacion negativa entre GE/CC y `fsi_2023`,
- asociacion fuerte entre `cc_2023` y `cpi_2023`, anticipando alta capacidad predictiva en ese target.

> Nota: en este notebook la ejecucion de graficos queda opcional para mantener tiempos de corrida controlados.

In [6]:
# run_script("EDA rev00.py")

## 7. Modelos supervisados

Se entrenan dos modelos de regresion para cada target:

- **Modelo 1 (GE):** prediccion de `ge_2023`.
- **Modelo 2 (CC):** prediccion de `cc_2023`.

Configuracion aplicada (alineada con scripts existentes):

- imputacion de faltantes por mediana,
- estandarizacion con `StandardScaler`,
- particion train/test 80/20 (`random_state=42`),
- algoritmos: Regresion Lineal y Random Forest (`n_estimators=500`).

Adicionalmente, el analisis post-entreno incorpora validacion cruzada K-Fold y comparacion de importancia de variables para ambos targets.

In [7]:
# run_script("modelo 1.py")
# run_script("modelo 2.py")
# run_script("Script maestro de analisis post entreno.py")

### 7.1 Registro de metricas (resultado reproducido)

| Modelo | Target | MAE | RMSE | R2 |
|---|---|---:|---:|---:|
| Regresion lineal | GE | 0.2693 | 0.3213 | 0.8878 |
| Random Forest | GE | 0.2399 | 0.3067 | 0.8978 |
| Regresion lineal | CC | 0.1195 | 0.1516 | 0.9760 |
| Random Forest | CC | 0.1200 | 0.1591 | 0.9735 |

**Validacion cruzada (Random Forest):**

- GE (`ge_2023`): R2 medio = **0.8940**, desviacion estandar = **0.0139**.
- CC (`cc_2023`): R2 medio = **0.9736**, desviacion estandar = **0.0064**.

Interpretacion breve:

- Para GE, Random Forest mejora ligeramente a la regresion lineal.
- Para CC, la regresion lineal supera marginalmente a Random Forest, coherente con una relacion predominantemente lineal con `cpi_2023`.

## 8. Analisis no supervisado

Se aplica `PCA & K-means no supervisado.py` sobre las mismas variables estructurales usadas en modelos supervisados.

Resultados principales observados:

- **PCA:**
  - PC1 explica **53.91%** de la varianza,
  - PC2 explica **11.44%**,
  - varianza acumulada (PC1+PC2): **65.35%**.
- **Seleccion de k (K-Means):**
  - Silhouette: k=2 (0.318), k=3 (0.295), k=4 (0.296), con perdida de interpretabilidad para k mayores.
  - Se mantiene **k=3** por equilibrio entre separacion e interpretacion institucional.
- **Perfiles de clusters (medias):**
  - Cluster 0: paises desarrollados (PIB, conectividad, GE/CC altos; fragilidad baja).
  - Cluster 1: paises fragiles (PIB, conectividad, GE/CC bajos; fragilidad alta).
  - Cluster 2: paises intermedios/en transicion.

Esto complementa el analisis supervisado al mostrar que los niveles de gobernanza emergen dentro de patrones estructurales coherentes.

In [8]:
# run_script("PCA & K-means no supervisado.py")

## 9. Resultados generados (data/processed)

In [9]:
produced = sorted([p.name for p in PROCESSED_DIR.glob("*") if p.is_file()])
for f in produced:
    print(f)

print("\nTotal archivos en processed:", len(produced))

.gitkeep
cpi_final.csv
dataset_maestro_2023.csv
fsi_final.csv
gdp_final.csv
resultado_clusters_pca_kmeans.csv
wdi_final.csv
wgi_cc_final.csv
wgi_gc_final.csv
wgi_ge_final.csv

Total archivos en processed: 10


## 10. Interpretacion de resultados

### Variables explicativas principales

- En **GE**, las variables con mayor peso en Random Forest son `cpi_2023` (0.692) y `fsi_2023` (0.188), seguidas a distancia por `gdp_per_capita`.
- En **CC**, `cpi_2023` domina casi por completo la prediccion (0.979), mientras el resto de variables tienen contribucion marginal.

### Rendimiento comparado de algoritmos

- **GE:** Random Forest supera ligeramente a la regresion lineal, sugiriendo componentes no lineales moderados.
- **CC:** la regresion lineal obtiene mejor desempeno que Random Forest, indicando una relacion mas lineal y directa.

### Coherencia sustantiva e institucional

- Los resultados son consistentes con el marco conceptual: la calidad institucional (corrupcion y fragilidad) explica mas que las variables socioeconomicas aisladas.
- PCA + K-Means confirma la estructura global: los paises con mayor desarrollo y menor fragilidad concentran mejores niveles de GE y CC.

### Implicaciones practicas

- Mejorar gobernanza requiere actuar sobre dimensiones institucionales (integridad publica y fortaleza del Estado), ademas de crecimiento economico.
- La segmentacion por clusters puede guiar estrategias diferenciadas de politica publica para paises desarrollados, fragiles e intermedios.

## 11. Conclusiones, limitaciones y trabajo futuro

### Conclusiones

- El pipeline completo (preprocesamiento, merge, modelado supervisado y no supervisado) es reproducible con las rutas estaticas del proyecto.
- `cc_2023` resulta mas predecible que `ge_2023`, con desempeno sobresaliente de ambos modelos y mejor ajuste de regresion lineal.
- `ge_2023` mantiene alto poder explicativo y mejora leve con Random Forest.
- Los resultados convergen en que gobernanza depende principalmente de factores institucionales (`cpi_2023`, `fsi_2023`) y, en segundo plano, de variables socioeconomicas.
- La segmentacion en tres clusters respalda la interpretacion de perfiles estructurales de paises (desarrollados, fragiles e intermedios).

### Limitaciones

- Analisis de corte transversal (2023): no captura dinamicas temporales.
- Cobertura desigual entre fuentes y necesidad de imputacion en algunas variables.
- Evaluacion concentrada en dos algoritmos base; no se comparan otros enfoques (XGBoost, SVR, etc.).
- Posible dependencia conceptual entre algunos indicadores institucionales (por ejemplo, CC y CPI).

### Trabajo futuro

- Extender a series temporales y panel de datos para analizar evolucion de gobernanza.
- Incorporar modelos adicionales y analisis de sensibilidad/robustez.
- Desarrollar explicabilidad avanzada (por ejemplo, SHAP) para interpretaciones locales y globales.
- Integrar resultados en un tablero de visualizacion para seguimiento por region/pais.